# **Guida a Python asyncio**

In questo notebook vengono ripresi e approfonditi i concetti legati alla programmazione asincrona e all'utilizzo di `asyncio` in Python già illustrati nel documento introduttivo.

## 1. **Tasks vs Coroutines**
### 1.1 **Le coroutines sono oggetti "pigri"**

Come già detto nell'introduzione, quando definiamo una funzione con `async def`, creiamo una **coroutine** (cooperative function). Ma attenzione: a differenza di quello che avviene con una funzione "standard", la chiamata di una coroutine non esegue il codice corrispondente!




In [ ]:
import asyncio

async def saluta(nome):
    print(f"Ciao {nome}!")
    return f"Saluto a {nome}"

# Chiamiamo la funzione
risultato = saluta("Mario")
print(type(risultato))  # <class 'coroutine'>
print(risultato)        # <coroutine object saluta at 0x...>

Nella cella precedente, l'istruzione `print("Ciao Mario!")` **non viene eseguita**! Chiamando `saluta("Mario")` si ottiene solo un oggetto coroutine.

Per eseguire una coroutine serve l'**event loop**:

In [ ]:
import asyncio

async def saluta(nome):
    print(f"Ciao {nome}!")
    return f"Saluto a {nome}"

# Modo 1: con asyncio.run() - crea l'event loop
#risultato = asyncio.run(saluta("Mario"))  #vale in ambiente NON Jupyter

risultato = await saluta("Mario")          #vale in ambiente Jupyter (notebook) perchè l'event loop esiste già
print(f"Risultato: {risultato}")

**Cosa succede:**
1. `asyncio.run(saluta("Mario"))` crea un nuovo event loop schedulando la coroutine `saluta("Mario")` nell'event loop;
3. event loop esegue la coroutine;
4. quando la coroutine termina, `asyncio.run()` restituisce il risultato;
5. event loop viene chiuso.

### 1.2. **`await` di una coroutine: esecuzione immediata e sequenziale**

Quando si utilizza `await` direttamente su una coroutine, l'event loop la esegue **immediatamente** in modo **sequenziale**.

In [ ]:
import asyncio
import time

async def operazione_a():
    print("A: inizio")
    await asyncio.sleep(2)
    print("A: fine")
    return "Risultato A"

async def operazione_b():
    print("B: inizio")
    await asyncio.sleep(2)
    print("B: fine")
    return "Risultato B"

async def main_sequenziale():
    print("=== Inizio main ===")
    inizio = time.time()

    # await diretto: esecuzione sequenziale
    ris_a = await operazione_a()  #← event loop esegue TUTTA operazione_a (avvia esecuzione e aspetta il termine)
    ris_b = await operazione_b()  #← POI esegue operazione_b (avvia esecuzione e aspetta il termine)

    fine = time.time()
    print(f"=== Fine in {fine - inizio:.1f}s ===")
    print(f"Risultati: {ris_a}, {ris_b}")

# asyncio.run(main_sequenziale())           #vale in ambiente NON Jupyter
await main_sequenziale()                    #vale in ambiente Jupyter (notebook) perchè l'event loop esiste già

#### Analisi temporale:

```
Tempo (secondi) →
0s:  main_sequenziale inizia
     ↓
     await operazione_a()  ← event loop passa il controllo a operazione_a
     
0s:  A: inizio
     await asyncio.sleep(2)  ← operazione_a si sospende
     
     [event loop aspetta 2 secondi - non fa altro!]
     
2s:  A: fine
     operazione_a termina e restituisce il controllo
     
     await operazione_b()  ← SOLO ORA event loop passa a operazione_b
     
2s:  B: inizio
     await asyncio.sleep(2)  ← operazione_b si sospende
     
     [event loop aspetta altri 2 secondi]
     
4s:  B: fine
     operazione_b termina
     
     Totale: 4 secondi (2 + 2)
```

**Punti chiave:**
- `await coroutine` dice all'event loop: "esegui questa coroutine ORA e aspetta che finisca";
- l'esecuzione è **sequenziale**: una dopo l'altra;
- è come il codice sincrono, ma con la possibilità di cedere il controllo durante le attese.

### 1.3. **task: schedulare una coroutine per l'esecuzione concorrente**

Un **task** è un *wrapper* che dice all'event loop: "schedula questa coroutine per l'esecuzione, ma non aspettare che finisca prima di fare altro".

In [ ]:
import asyncio
import time

async def operazione_a():
    print("A: inizio")
    await asyncio.sleep(2)
    #time.sleep(2)
    print("A: fine")
    return "Risultato A"

async def operazione_b():
    print("B: inizio")
    await asyncio.sleep(2)
    #time.sleep(2)
    print("B: fine")
    return "Risultato B"

async def main_concorrente():
    print("=== Inizio main ===")
    inizio = time.time()

    # Creiamo i task: scheduliamo le coroutine
    task_a = asyncio.create_task(operazione_a())  # ← Schedulata subito
    task_b = asyncio.create_task(operazione_b())  # ← Schedulata subito

    print("Task creati, ora aspettiamo...")
    ris_b =""
    ris_a =""

    # Aspettiamo i risultati
    ris_b = await task_b
    ris_a = await task_a
    fine = time.time()
    print(f"=== Fine in {fine - inizio:.1f}s ===")
    print(f"Risultati: {ris_a}, {ris_b}")
    await asyncio.sleep(3)

# asyncio.run(main_concorrente())   # No Jupyter
await main_concorrente()            # Solo in Jupyter

#### Analisi temporale:

```
Tempo (secondi) →
0s:  main_concorrente inizia
     
     task_a = asyncio.create_task(operazione_a())
     ↓
     Event loop SCHEDULA operazione_a nella coda (non esegue ancora!)
     
     task_b = asyncio.create_task(operazione_b())
     ↓
     Event loop SCHEDULA operazione_b nella coda
     
     print("Task creati...")  ← Esegue questa riga
     
     await task_a  ← main si sospende
     ↓
     Event loop esegue task_a dalla coda
     
0s:  A: inizio
     await asyncio.sleep(2)  ← task_a si sospende
     ↓
     Event loop esegue task_b dalla coda
     
0s:  B: inizio  
     await asyncio.sleep(2)  ← task_b si sospende
     
     [event loop gestisce entrambi i task in attesa]
     
2s:  A: fine  ← operazione_a completa
     B: fine  ← operazione_b completa
     
     main_concorrente riprende
     
     Totale: 2 secondi (concorrenti, non 2+2!)
```

### 1.4. **La differenza fondamentale: `await` coroutine vs `await` task**

#### Il momento cruciale: quando viene eseguita la coroutine?

Vediamo un esempio che mostra chiaramente la differenza:

In [ ]:
import asyncio

async def operazione_lunga(nome):
    print(f"  [{nome}] Inizio esecuzione")
    await asyncio.sleep(2)
    print(f"  [{nome}] Fine esecuzione")
    return f"Risultato {nome}"

async def main():
    print("=== Inizio ===")

    task_b = asyncio.create_task(operazione_lunga("task-B"))    #schedula operazione_lunga() come task-B
    risultato_a = await operazione_lunga("coroutine-A")         #await operazione_lunga() come coroutine-A
    risultato_b = await task_b                                  #attendi termine task-B

    print(f"\nRisultati: {risultato_a}, {risultato_b}")
    print("=== Fine ===")

# asyncio.run(main())     #no Jupyter
await main()              #solo in Jupyter

**Osservazione importante:** task-B viene eseguito **dopo** che coroutine-A inizia, anche se è stato schedulato prima!

#### Perché succede questo?

1. `create_task()` **aggiunge il task alla coda** dell'event loop;
2. il task viene **effettivamente eseguito** solo quando l'event loop ha un momento libero;
3. in questo caso, il momento libero arriva quando Coroutine-A si sospende su `await sleep(2)`.

### Tabella riassuntiva:

| Aspetto | `await coroutine` | `await task` |
|---------|-------------------|-------------|
| **Quando viene eseguita** | Immediatamente | Quando l'event loop è libero |
| **Tipo di esecuzione** | Sequenziale | Concorrente |
| **Comportamento** | Event loop esegue la coroutine subito | Event loop aggiunge il task alla coda |
| **Blocco** | Blocca finché la coroutine non finisce | Altri task possono partire nel frattempo |
| **Uso tipico** | Operazioni che devono completarsi prima di proseguire | Operazioni che possono procedere in parallelo |

### 1.5. **Dare "respiro" all'event loop**

Se vogliamo che un task schedulato inizi immediatamente, dobbiamo dare un momento di "respiro" all'event loop:

In [ ]:
import asyncio

async def operazione_lunga(nome):
    print(f"  [{nome}] Inizio")
    await asyncio.sleep(2)
    print(f"  [{nome}] Fine")
    return f"Risultato {nome}"

async def main():
    print("=== Con await asyncio.sleep(0) ===")

    task_b = asyncio.create_task(operazione_lunga("Task-B"))  #schedula operazione_lunga() come task-B

    #questo dà un "momento di respiro" all'event loop
    await asyncio.sleep(0)  # ← cede il controllo all'event loop!

    risultato_a = await operazione_lunga("Coroutine-A")     #await operazione_lunga() come coroutine-A
    risultato_b = await task_b                              #attendi termine task-B

    print(f"\nRisultati: {risultato_a}, {risultato_b}")

# asyncio.run(main())   #no Jupyter
await main()            #solo in Jupyter

**Ora sì che Task-B inizia prima!** Perché `await asyncio.sleep(0)` dice: "event loop, esegui i task in coda".

### 1.6. **Confronto diretto: sequenziale vs concorrente**

Esempio finale che mostra chiaramente la differenza di prestazioni:

In [ ]:
import asyncio
import time

async def scarica_file(nome, secondi):
    print(f"[{time.strftime('%H:%M:%S')}] {nome}: inizio download")
    await asyncio.sleep(secondi)
    print(f"[{time.strftime('%H:%M:%S')}] {nome}: download completato")
    return f"Dati di {nome}"

# VERSIONE 1: SEQUENZIALE
async def versione_sequenziale():
    print("\n=== VERSIONE SEQUENZIALE (await coroutine) ===")
    inizio = time.time()

    # Esecuzione sequenziale --> await su coroutines
    dati1 = await scarica_file("file1.txt", 2)
    dati2 = await scarica_file("file2.txt", 2)
    dati3 = await scarica_file("file3.txt", 2)

    print(f"Tempo totale: {time.time() - inizio:.1f}s\n")

# VERSIONE 2: CONCORRENTE
async def versione_concorrente():
    print("\n=== VERSIONE CONCORRENTE (await task) ===")
    inizio = time.time()

    # Creazione dei tasks (schedulati immediatamente)
    task1 = asyncio.create_task(scarica_file("file1.txt", 2))
    task2 = asyncio.create_task(scarica_file("file2.txt", 2))
    task3 = asyncio.create_task(scarica_file("file3.txt", 2))

    # Attesa dei risultati --> await su tasks
    dati1 = await task1
    dati2 = await task2
    dati3 = await task3

    print(f"Tempo totale: {time.time() - inizio:.1f}s\n")

# Esecuzione di entrambe le versioni
# asyncio.run(versione_sequenziale())   #no Jupyter
# asyncio.run(versione_concorrente())   #no Jupyter
await versione_sequenziale()            #solo in Jupyter
await versione_concorrente()            #solo in Jupyter

**Risultato atteso:**
- Versione sequenziale: 6 secondi (2+2+2)
- Versione concorrente: 2 secondi (tutti insieme)

## 2. **Altre funzionalità sui Tasks**

### 2.1. **Visualizzare lo stato dei Tasks**

In [ ]:
import asyncio

async def operazione_lenta():
    print("Operazione iniziata")
    await asyncio.sleep(2)
    print("Operazione completata")
    return 42

async def main():
    # Creazione del task
    task = asyncio.create_task(operazione_lenta())

    # Visualizzazione dello stato del task (terminato/cancellato)
    print(f"Task creato - stato: {task.done()}")  # False
    print(f"Task cancellato? {task.cancelled()}")  # False

    # tempo all'event loop
    await asyncio.sleep(0.1)
    print(f"Dopo 0.1s - stato: {task.done()}")  # False

    # Attesa del completamento
    risultato = await task
    print(f"Task completato - stato: {task.done()}")  # True
    print(f"Risultato: {risultato}")
    print(f"Task cancellato? {task.cancelled()}")  # False

# asyncio.run(main())     #no jupyter
await main()              #solo in Jupyter

### 2.2. **await su Task già completato**

Cosa succede se si esegue `await` su un task che è già terminato?

In [ ]:
import asyncio
import time

async def operazione():
    print(f"  {time.time():.2f} - Operazione in corso")
    await asyncio.sleep(1)
    print(f"  {time.time():.2f} - Operazione completata")
    return "Risultato"

async def main():
    inizio = time.time()
    print(f"{time.time():.2f} - Creo task")

    task = asyncio.create_task(operazione())

    # Aspettiamo che completi
    await asyncio.sleep(2)

    print(f"\n{time.time():.2f} - Task completato, ora faccio await")
    risultato = await task  # ← Questo è istantaneo!
    print(f"{time.time():.2f} - await completato (istantaneo)")

    print(f"\nTempo totale: {time.time() - inizio:.2f}s")

# asyncio.run(main())     #no Jupyter
await main()              #solo in Jupyter

**Risposta:** Se il task è già completato, `await task` restituisce **immediatamente** il risultato senza sospendere la coroutine!

Inoltre, è possibile fare `await` su un task completato **infinite volte**, ottenendo sempre lo stesso risultato:

In [ ]:
import asyncio

async def operazione():
    await asyncio.sleep(1)
    return "Risultato"

async def main():
    task = asyncio.create_task(operazione())

    # Prima attesa
    risultato1 = await task
    print(f"Prima await: {risultato1}")

    # Seconda attesa (task già completato)
    risultato2 = await task
    print(f"Seconda await: {risultato2}")

    # Terza attesa
    risultato3 = await task
    print(f"Terza await: {risultato3}")

# asyncio.run(main())     #no Jupyter
await main()              #solo in Jupyter

### 2.3. **Accedere al risultato senza await: il metodo result()**

Se un task è già completato, si può ottenere il risultato che ha prodotto direttamente senza `await`:

In [ ]:
import asyncio

async def operazione():
    await asyncio.sleep(1)
    return "Risultato"

async def main():
    task = asyncio.create_task(operazione())

    #prima che il task si sia completato
    print(f"1. done={task.done()}")
    try:
        risultato = task.result()  # ← errore!
    except asyncio.InvalidStateError as e:
        print(f"   Errore: {e}")

    #attesa sufficiente a completare il task
    await asyncio.sleep(1.5)

    #dopo che il task si è completato
    print(f"\n2. done={task.done()}")
    risultato = task.result()  # ← OK! nessun await necessario
    print(f"   Risultato con .result(): {risultato}")

    #await funziona comunque
    risultato2 = await task
    print(f"   Risultato con await: {risultato2}")

# asyncio.run(main())     #no Jupyter
await main()              #solo in Jupyter

## 3. **Riepilogo dei meccanismi fondamentali**

#### **Coroutine**
- oggetto creato chiamando una funzione `async def`;
- non viene eseguita finché non è schedulata nell'event loop;
- può sospendersi con `await`.

#### **Event Loop**
- esegue un pezzo di codice alla volta (single-threaded);
- tiene una coda di task pronti per l'esecuzione;
- quando un task si sospende, passa al successivo.

#### **await coroutine**
- event loop esegue la coroutine **subito**;
- comportamento **sequenziale**;
- se la coroutine si sospende, il controllo torna all'event loop;

#### **create_task() + await task**
- `create_task()` **schedula** il task nella coda (non lo esegue subito!);
- il task viene eseguito quando l'event loop è libero;
- comportamento **concorrente**;
- altri task possono essere creati ed eseguiti contemporaneamente:

#### **await su task completato**
- se il task è già completato, `await task` restituisce il risultato **immediatamente**;
- non c'è sospensione, non c'è attesa;
- si può fare `await` su un task completato un numero indeterminato di volte;
- si può usare `.result()` per ottenere il risultato senza `await` (se il task è completato).

## 4. **Scenario pratico: Simulazione download di file**

L'esempio seguente simula il download di 5 pagine web con tempi casuali:

In [ ]:
import asyncio
import random
import time

async def scarica_pagina(url, numero):
    durata = random.randint(1, 3)
    print(f"[{time.strftime('%H:%M:%S')}] Download {numero}: {url} iniziato (stima: {durata}s)")

    await asyncio.sleep(durata)

    print(f"[{time.strftime('%H:%M:%S')}] Download {numero}: {url} completato")
    return f"Contenuto di {url}"

async def main_sequenziale():
    """Versione sequenziale - lenta"""
    print("\n=== DOWNLOAD SEQUENZIALE ===")
    urls = [
        "http://example.com/page1",
        "http://example.com/page2",
        "http://example.com/page3",
        "http://example.com/page4",
        "http://example.com/page5"
    ]

    inizio = time.time()

    risultati = []
    for i, url in enumerate(urls, 1):
        contenuto = await scarica_pagina(url, i)
        risultati.append(contenuto)

    tempo_totale = time.time() - inizio
    print(f"\n✓ Completati {len(risultati)} download in {tempo_totale:.1f}s")
    return tempo_totale

async def main_concorrente():
    """Versione concorrente - veloce"""
    print("\n=== DOWNLOAD CONCORRENTE ===")
    urls = [
        "http://example.com/page1",
        "http://example.com/page2",
        "http://example.com/page3",
        "http://example.com/page4",
        "http://example.com/page5"
    ]

    inizio = time.time()

    # Creiamo tutti i task contemporaneamente
    tasks = []
    for i, url in enumerate(urls, 1):
        task = asyncio.create_task(scarica_pagina(url, i))
        tasks.append(task)

    # Aspettiamo che tutti completino
    risultati = await asyncio.gather(*tasks)

    tempo_totale = time.time() - inizio
    print(f"\n✓ Completati {len(risultati)} download in {tempo_totale:.1f}s")
    return tempo_totale

async def confronto():
    tempo_seq = await main_sequenziale()
    tempo_conc = await main_concorrente()

    risparmio = tempo_seq - tempo_conc
    percentuale = (risparmio / tempo_seq) * 100

    print(f"\n{'='*50}")
    print(f"CONFRONTO:")
    print(f"  Sequenziale:  {tempo_seq:.1f}s")
    print(f"  Concorrente:  {tempo_conc:.1f}s")
    print(f"  Risparmio:    {risparmio:.1f}s ({percentuale:.0f}%)")
    print(f"{'='*50}")

# asyncio.run(confronto())    # No Colab
await confronto()             # Solo in Colab

## 5. **Esercizi di verifica**

### 5.1: Prevedere l'output

Prima di eseguire il codice, si provi a prevedere:
- quanto sarà il tempo di esecuzione?
- in che ordine appariranno i messaggi?

In [ ]:
import asyncio
import time

async def conta(nome, num):
    for i in range(num):
        print(f"{nome}: {i}")
        await asyncio.sleep(0.5)
    return f"{nome} ha finito"

async def main():
    inizio = time.time()

    r1 = await conta("A", 3)
    r2 = await conta("B", 3)

    print(f"\nTempo: {time.time() - inizio:.1f}s")
    print(f"Risultati: {r1}, {r2}")

# asyncio.run(main())     #no Jupyter
await main()              #solo in jupyter

### 5.2: Modifica in direzione concorrente

Se si modifica il codice precedente usando `create_task()`, cosa cambia? Quale sarà la risposta alle stesse domande dell'eserczio 5.1?

In [ ]:
import asyncio
import time

async def conta(nome, num):
    for i in range(num):
        print(f"{nome}: {i}")
        await asyncio.sleep(0.5)
    return f"{nome} ha finito"

async def main():
    inizio = time.time()

    # TODO: Modifica qui per usare create_task()
    # task_a = ...
    # task_b = ...
    # r1 = ...
    # r2 = ...

    print(f"\nTempo: {time.time() - inizio:.1f}s")
    # print(f"Risultati: {r1}, {r2}")

# asyncio.run(main())     #no Jupyter
await main()              #solo in jupyter

### 5.3: Analisi dell'ordine di esecuzione

Eseguire il codice seguente e analizzare l'ordine di esecuzione:

In [2]:
import asyncio
import time

async def operazione(nome, tempo):
    print(f"→ [{nome}] Inizio")
    #await asyncio.sleep(0.5)
    await asyncio.sleep(tempo)
    print(f"← [{nome}] Fine")

async def test():
    print("\n=== Inizio test ===")
    sTime = time.time()

    task1 = asyncio.create_task(operazione("Task1", 1.5))
    print("Task1 schedulato")

    task2 = asyncio.create_task(operazione("Task2", 1.0))
    print("Task2 schedulato")

    #await task1

    await operazione("Coroutine3", 2)
    print("Coroutine3 completata")

    await task1
    await task2
    print("Task1 e Task2 completati")

    print(f"Totale: {time.time() - sTime:.1f}s")
    print("=== Fine test ===")

# asyncio.run(test())     #no Jupyter
await test()              #solo in Jupyter


=== Inizio test ===
Task1 schedulato
Task2 schedulato
→ [Coroutine3] Inizio
→ [Task1] Inizio
→ [Task2] Inizio
← [Task2] Fine
← [Task1] Fine
← [Coroutine3] Fine
Coroutine3 completata
Task1 e Task2 completati
Totale: 2.0s
=== Fine test ===


**Domande:**
1. In che ordine vengono stampati i messaggi "Inizio"?
2. Perché Coroutine3 inizia prima di Task1 e Task2?
3. Cosa succederebbe se aggiungessimo `await asyncio.sleep(0)` dopo aver creato i task?
4. Cosa cambia invece "scommentando" la riga 20?

### 5.4: Task con diversi tempi di esecuzione

Si creino 5 task che impiegano tempi diversi (1, 2, 3, 4, 5 secondi) e si osservi l'ordine di completamento:

In [ ]:
import asyncio
import time

async def operazione_temporizzata(nome, durata):
    print(f"[{time.strftime('%H:%M:%S')}] {nome} inizia (durata: {durata}s)")
    await asyncio.sleep(durata)
    print(f"[{time.strftime('%H:%M:%S')}] {nome} completato")
    return f"Risultato di {nome}"

async def main():

    inizio = time.time()

    # TODO: Crea 5 task con durate 1, 2, 3, 4, 5 secondi
    # Suggerimento: usa un loop

    # TODO: Attendi tutti i task
    # Suggerimento: usa asyncio.gather()

    print(f"\nTempo totale: {time.time() - inizio:.1f}s")

# asyncio.run(main())   #no jupyter
await main()            #solo in Jupyter

## 6. **Pattern comuni con AsyncIO**

### 6.1: **Eseguire task e continuare**

A volte può essere necessario avviare un task senza aspettare immediatamente che termini:

In [ ]:
import asyncio

async def operazione_lunga():
    print("Operazione lunga iniziata...")
    await asyncio.sleep(3)
    print("Operazione lunga completata!")
    return "Risultato importante"

async def operazione_veloce():
    print("Operazione veloce...")
    await asyncio.sleep(0.5)
    return "Fatto subito"

async def main():
    # Avvia l'operazione lunga ma non aspettare
    task_lungo = asyncio.create_task(operazione_lunga())

    # Fai altre cose nel frattempo
    print("Faccio altre cose mentre l'operazione lunga procede...")
    risultato_veloce = await operazione_veloce()
    print(f"Completato: {risultato_veloce}")

    # Ora aspetta l'operazione lunga
    print("\nOra aspetto l'operazione lunga...")
    risultato_lungo = await task_lungo
    print(f"Completato: {risultato_lungo}")

# asyncio.run(main())     #no Jupyter
await main()              #solo in Jupyter

### 6.2: **Timeout su operazioni**

Usare `asyncio.wait_for()` per impostare un timeout:

In [ ]:
import asyncio

async def operazione_lenta():
    print("Inizio operazione lenta...")
    await asyncio.sleep(5)
    return "Completato"

async def main():
    try:
        # Timeout di 2 secondi
        risultato = await asyncio.wait_for(operazione_lenta(), timeout=2.0)
        print(f"Risultato: {risultato}")
    except asyncio.TimeoutError:
        print("Operazione troppo lenta! Timeout raggiunto.")

# asyncio.run(main())     #no Jupyter
await main()              #solo in Jupyter

### 6.3: **Gestione di eccezioni nei task**

Come gestire errori in task concorrenti:

In [ ]:
import asyncio

async def operazione_ok():
    await asyncio.sleep(1)
    return "Successo"

async def operazione_errore():
    await asyncio.sleep(1)
    raise ValueError("Errore simulato!")

async def main():
    task1 = asyncio.create_task(operazione_ok())
    task2 = asyncio.create_task(operazione_errore())
    task3 = asyncio.create_task(operazione_ok())

    # gather con return_exceptions=True continua anche se ci sono errori
    risultati = await asyncio.gather(task1, task2, task3, return_exceptions=True)

    for i, risultato in enumerate(risultati, 1):
        if isinstance(risultato, Exception):
            print(f"Task {i}: Errore - {risultato}")
        else:
            print(f"Task {i}: {risultato}")

# asyncio.run(main())     # No Colab
await main()              # Solo in Colab

## 7. **Recap finale: Regole d'oro**

### ✅ Cosa fare:

1. **Usa `create_task()` per operazioni concorrenti**
   ```python
   task1 = asyncio.create_task(op1())
   task2 = asyncio.create_task(op2())
   await asyncio.gather(task1, task2)
   ```

2. **Usa `await coroutine` per operazioni sequenziali**
   ```python
   risultato1 = await op1()
   risultato2 = await op2()  # Solo dopo che op1 finisce
   ```

3. **Usa `asyncio.gather()` per aspettare task multipli**
   ```python
   risultati = await asyncio.gather(task1, task2, task3)
   ```

4. **Controlla lo stato dei task con `.done()`**
   ```python
   if task.done():
       risultato = task.result()
   ```

### ❌ Cosa evitare:

1. **Non chiamare coroutine senza await o create_task**
   ```python
   # ❌ SBAGLIATO
   operazione()  # Crea solo un oggetto, non esegue!
   
   # ✅ CORRETTO
   await operazione()
   # oppure
   task = asyncio.create_task(operazione())
   ```

2. **Non usare `time.sleep()` in codice async**
   ```python
   # ❌ SBAGLIATO - blocca tutto!
   time.sleep(1)
   
   # ✅ CORRETTO - permette altri task
   await asyncio.sleep(1)
   ```

3. **Non dimenticare await davanti alle coroutine**
   ```python
   # ❌ SBAGLIATO
   risultato = operazione_async()  # Ottieni un oggetto coroutine!
   
   # ✅ CORRETTO
   risultato = await operazione_async()
   ```

## 8. **Conclusioni**

Abbiamo visto i meccanismi fondamentali di AsyncIO:

1. **Coroutine**: funzioni `async def` che possono sospendersi
2. **Event Loop**: il coordinatore che gestisce l'esecuzione
3. **await**: sospende l'esecuzione e cede il controllo
4. **Task**: wrapper per esecuzione concorrente
5. **Differenza cruciale**: `await coroutine` (sequenziale) vs `create_task() + await task` (concorrente)
6. **Schedulazione**: `create_task()` schedula ma non esegue immediatamente

### Quando usare AsyncIO:
- ✅ Operazioni I/O-bound (rete, database, file)
- ✅ Gestione di molte connessioni simultanee
- ✅ Download/upload multipli
- ✅ API calls concorrenti

### Quando NON usare AsyncIO:
- ❌ Operazioni CPU-intensive (calcoli pesanti) → usare `multiprocessing`
- ❌ Codice semplice sequenziale dove non serve concorrenza

---